# 360 City Analyser on Colab

Runtime > Change runtime type > **T4 GPU** (or better). Run the cells in order.
This notebook has not been run on Colab yet: if a cell fails, report the error.

## 1. Drive, repository and environment

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = '/content/drive/MyDrive/city-analyser'      # data, models and caches persist here
os.makedirs(WORK + '/data', exist_ok=True); os.makedirs(WORK + '/models', exist_ok=True)

In [ ]:
# Private repository: store a read-only GitHub token in Colab > Secrets as GITHUB_TOKEN.
# For a public repository the URL needs no token. Alternative: upload a zip and unpack it into /content/repo.
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
!rm -rf /content/repo && git clone https://{token}@github.com/JSempereH/360_city_analyser.git /content/repo

In [ ]:
%cd /content/repo
!pip -q install uv
!uv sync --extra cu126
# data/ and models/ live on Drive; the repository sees them as usual
!rm -rf data models && ln -s {WORK}/data data && ln -s {WORK}/models models

In [ ]:
import os
os.environ['BUILDING_ANALYSIS_MODEL'] = 'high'                       # for a 16 GB T4; use 'balanced' if memory runs out
os.environ['BUILDING_ANALYSIS_STAGE_CACHE_DIR'] = WORK + '/stage-cache'
os.environ['BUILDING_ANALYSIS_OSM_CACHE_DIR'] = WORK + '/osm-cache'   # Colab can reach Overpass; the cache avoids repeating requests
!uv run city-analyser-models

## 2. Data

Option A: copy a dataset folder from your laptop (for example `data/valencia-centro`) to `MyDrive/city-analyser/data/`.

Option B: download it from Mapillary (token in Colab > Secrets as `MAPILLARY_ACCESS_TOKEN`). `--tiled-bbox` accepts an area of any size
and queries it tile by tile; rerunning the same command skips the files already downloaded.

In [ ]:
from google.colab import userdata
os.environ['MAPILLARY_ACCESS_TOKEN'] = userdata.get('MAPILLARY_ACCESS_TOKEN')
DATASET = 'valencia-centro'   # change to your dataset name
# min_lon,min_lat,max_lon,max_lat: replace with your own area (start small: a few hundred panoramas)
BBOX = '-0.3800,39.4600,-0.3600,39.4750'
!uv run city-analyser-download --tiled-bbox {BBOX} --output data/{DATASET} --max-images 500 --image-size 2048 --workers 8

In [ ]:
!ls data/{DATASET} | head

## 3. Analyze the city: resumable, with visible progress

The analysis runs in the background. You can close this notebook, restart it or lose the session: running the cell again continues where it
stopped (panoramas already finished are recognized from `batch-ledger.jsonl` in under a second). Transient errors are retried, panoramas that
keep failing are retried by the next run, and a panorama with bad data is recorded and not repeated.

In [ ]:
import subprocess
log = open(f'{WORK}/analyze-{DATASET}.log', 'a')
job = subprocess.Popen(['uv', 'run', 'city-analyser-analyze', DATASET, '--skip-low-quality', '--report', f'{WORK}/report-{DATASET}.json'],
                       stdout=log, stderr=subprocess.STDOUT, cwd='/content/repo')

Progress panel: bar, estimated time left, a map of the city colored by status and the latest errors. It refreshes by itself and stops when the
analysis ends. You can interrupt this cell without stopping the analysis.

In [ ]:
import sys
sys.path.insert(0, '/content/repo/src')   # run_progress only needs Pillow, which Colab already has
from city_analyser.run_progress import watch
from pathlib import Path
watch(Path('/content/repo/data'), DATASET, every=15)

In [ ]:
!tail -5 {WORK}/analyze-{DATASET}.log
print('process:', 'running' if job.poll() is None else f'finished with code {job.returncode} (0 = ok, 1 = aborted after consecutive failures, 130 = interrupted)')

The report written to Drive holds per-panorama timings and peak memory: these are the first real GPU numbers for the project.

To split the work, run only the model stages here with `--perception-only` (this fills the model cache on Drive), then run
`city-analyser-analyze` on your laptop with the same cache folder (each panorama then takes about a second).

## 4. Building index, crops and attributes (OSM tags)

In [ ]:
!uv run city-analyser-index {DATASET} --select
!uv run city-analyser-crops {DATASET} --per-building 3 --mask-others
!uv run city-analyser-attributes {DATASET}
!uv run python -m evaluation.attributes --dataset {DATASET} --provider osm-tag-prior | head -40

In [ ]:
# Per-building table (GeoJSON, CSV, GeoParquet when pyarrow is installed) and coverage report
!uv sync --extra cu126 --extra parquet
!uv run city-analyser-export {DATASET} --provider osm-tag-prior
!head -40 data/{DATASET}/exports/coverage.json

## 5. Attributes with a vision-language model (optional)

The model is served with vLLM in a separate environment (do not mix it with the project's torch) and the command line tool calls it on `localhost`.
Check that the model exists and that vLLM supports it before launching it. On a T4, use small models (1B to 4B) in fp16.

In [ ]:
!pip -q install uv
!uv venv /content/vllm-env && uv pip install --python /content/vllm-env/bin/python vllm
import subprocess
MODEL = 'Qwen/Qwen3-VL-2B-Instruct'   # example: verify the exact name on Hugging Face
server = subprocess.Popen(['/content/vllm-env/bin/vllm', 'serve', MODEL, '--dtype', 'half', '--max-model-len', '4096', '--port', '8000'],
                          stdout=open('/content/vllm.log', 'w'), stderr=subprocess.STDOUT)

In [ ]:
!tail -5 /content/vllm.log   # wait until you see 'Application startup complete'

In [ ]:
!uv run city-analyser-attributes {DATASET} --provider vlm --vlm-url http://127.0.0.1:8000 --vlm-model {MODEL}
!uv run python -m evaluation.attributes --dataset {DATASET} --provider vlm_{MODEL.replace('/', '_')} | head -60

## 6. Results

Everything stays on Drive (`data/<dataset>/analysis`, `crops`, `attributes`, `exports`, `building-index.json`). Download those folders and
open the viewer on your laptop with `make viewer`.